In [3]:
# base libraries
import sqlite3
import pandas as pd
from pathlib import Path
import os
from difflib import SequenceMatcher
import geonamescache

# NLP 
import spacy
from transformers import pipeline

# topic modeling
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer


In [4]:
# ============================================================
# 0. GLOBAL CONFIG
# ============================================================

DB_PATH = "gnews_articles_from2023.db"
ARTICLES_TABLE = "articles"
ARTICLE_LIMIT = 700

OUTPUT_EVENTS = "processed_conflict_events.csv"
OUTPUT_ARTICLES = "processed_conflict_articles.csv"

SIMILARITY_THRESHOLD = 0.8


In [5]:
# ============================================================
# 1. GERMANY LOCATION DICTIONARY
# ============================================================
gc = geonamescache.GeonamesCache()
cities = gc.get_cities()

GERMAN_LOCATIONS = {
    city["name"].lower()
    for city in cities.values()
    if city["countrycode"] == "DE"
}

GERMAN_EXTRAS = {
    "deutschland", "germany", "bundesrepublik", "berlin", "münchen", "munich", 
    "hamburg", "köln", "cologne", "frankfurt", "stuttgart", "düsseldorf", 
    "dortmund", "essen", "leipzig", "bremen", "dresden", "hannover", "nürnberg", 
    "duisburg", "bochum", "wuppertal", "bielefeld", "bonn", "münster", "karlsruhe",
    "mannheim", "augsburg", "wiesbaden", "gelsenkirchen", "mönchengladbach", 
    "braunschweig", "chemnitz", "aachen", "kiel", "halle", "magdeburg", "freiburg", 
    "oberhausen", "lübeck", "erfurt", "mainz", "rostock", "kassel", "hagen", 
    "saarbrücken", "hamm", "potsdam", "ludwigshafen", "oldenburg", "leverkusen", 
    "osnabrück", "solingen", "heidelberg", "herne", "neuss", "darmstadt", "paderborn", 
    "remscheid", "regensburg", "ingolstadt", "würzburg", "wolfsburg", "fürth", "ulm", "offenbach"
}
GERMAN_LOCATIONS.update(GERMAN_EXTRAS)

In [6]:
# ============================================================
# 2. MODEL INITIALIZATION
# ============================================================
print("Initializing NLP Models... (This may take a few minutes)")

# NLP for German NER (to detect domestic vs international)
try:
    nlp = spacy.load("de_core_news_sm")
except:
    print("Downloading spacy model...")
    os.system("python -m spacy download de_core_news_sm")
    nlp = spacy.load("de_core_news_sm")

# Translation Pipeline (DE -> EN)
translator = pipeline("translation_de_to_en", model="Helsinki-NLP/opus-mt-de-en")

# Emotion Analysis (English)
emotion_pipeline = pipeline("text-classification", model="finiteautomata/bertweet-base-emotion-analysis")

acled_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)



Initializing NLP Models... (This may take a few minutes)


Device set to use mps:0
Device set to use mps:0
Device set to use mps:0


In [7]:
# ============================================================
# 3. DATA LOADING
# ============================================================

def load_articles(db_path, limit):
    conn = sqlite3.connect(db_path)
    query = f"""
        SELECT publishedAt, title, description, content, source_name
        FROM {ARTICLES_TABLE}
        LIMIT {limit}
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    df["published_at_dt"] = pd.to_datetime(df["publishedAt"])
    df["published_date"] = df["published_at_dt"].dt.date
    df["published_month"] = df["published_at_dt"].dt.to_period("M").astype(str)

    return df

# ============================================================
# 4. GEOGRAPHY DETECTION
# ============================================================

def detect_domestic(text_de):
    """
    Enhanced location detection combining spaCy NER and keyword matching.
    """
    if not text_de:
        return False, ""
        
    doc = nlp(text_de)
    
    found_locs = []
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC"]:
            clean_loc = ent.text.strip().lower().rstrip(".,")
            found_locs.append(clean_loc)
    

    words = text_de.lower().split()
    for word in words:
        clean_word = word.strip(".,:;()!\"")
        if clean_word in GERMAN_EXTRAS:
            found_locs.append(clean_word)
            
    unique_locs = set(found_locs)
    
    is_domestic = any(loc in GERMAN_LOCATIONS for loc in unique_locs)
    
    return is_domestic, ", ".join(unique_locs)

# ============================================================
# 5. SIMILARITY FUNCTION
# ============================================================

def title_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# ============================================================
# 6. TRANSLATION (BATCH)
# ============================================================

def batch_translate(texts):
    results = []
    for text in texts:
        if not isinstance(text, str) or text.strip() == "":
            results.append("")
            continue
        translated = translator(text[:512])[0]["translation_text"]
        results.append(translated)
    return results

# ============================================================
# 7. SENTIMENT MAPPING
# ============================================================

def extract_sentiment(text):
    try:
        res = emotion_pipeline(text[:512])[0]
        label, score = res["label"], res["score"]
    except:
        return "neutral", 0.0

    if label in ["joy", "optimism"]:
        return label, score
    if label in ["anger", "sadness"]:
        return label, -score
    return label, 0.0

# ============================================================
# 8. ACLED EVENT CLASSIFICATION
# ============================================================

ACLED_EVENT_TYPES = [
    "Protests",
    "Battles",
    "Strategic developments",
    "Violence against civilians",
    "Riots",
    "Explosions/Remote violence"
]

def classify_acled_event(text):
    try:
        res = acled_classifier(text, ACLED_EVENT_TYPES)
        return res["labels"][0]
    except:
        return "Other"


In [8]:
# ============================================================
# 9. MAIN PIPELINE
# ============================================================

def run_pipeline():

    print("Loading articles...")
    df = load_articles(DB_PATH, ARTICLE_LIMIT)
    if df.empty:
        print("No data found.")
        return

    # --------------------------------------------------------
    # Step A: Geography
    # --------------------------------------------------------
    print("Detecting geographic scope and extracting locations...")
    geo_results = df.apply(
        lambda x: detect_domestic(f"{x['title']} {x.get('description', '')}"),
        axis=1
    )
    df["is_domestic"] = geo_results.apply(lambda x: x[0])
    df["detected_locations"] = geo_results.apply(lambda x: x[1])

    # --------------------------------------------------------
    # Step B: Event-level deduplication
    # --------------------------------------------------------
    print("Clustering articles into events...")

    df = df.sort_values(by=["published_date", "title"])
    df["article_cluster_id"] = range(len(df))
    df["is_duplicate"] = False

    for i in range(1, len(df)):
        same_day = df.iloc[i]["published_date"] == df.iloc[i - 1]["published_date"]
        if same_day:
            sim = title_similarity(
                df.iloc[i]["title"],
                df.iloc[i - 1]["title"]
            )
            if sim >= SIMILARITY_THRESHOLD:
                df.iloc[i, df.columns.get_loc("is_duplicate")] = True
                df.iloc[i, df.columns.get_loc("article_cluster_id")] = (
                    df.iloc[i - 1]["article_cluster_id"]
                )

    # --------------------------------------------------------
    # Step C: ARTICLE-LEVEL EXPORT (optional)
    # --------------------------------------------------------
    df.to_csv(OUTPUT_ARTICLES, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Step D: EVENT-LEVEL REDUCTION
    # --------------------------------------------------------
    print("Reducing to event-level dataset...")

    df_events = (
        df.sort_values(by=["article_cluster_id", "published_at_dt"])
          .groupby("article_cluster_id")
          .first()
          .reset_index()
    )

    # --------------------------------------------------------
    # Step E: Translation
    # --------------------------------------------------------
    print("Translating event texts...")
    df_events["title_en"] = batch_translate(df_events["title"].tolist())
    df_events["description_en"] = batch_translate(df_events["description"].tolist())
    df_events["analysis_text_en"] = (
        df_events["title_en"] + " " + df_events["description_en"]
    )

    # --------------------------------------------------------
    # Step F: Sentiment
    # --------------------------------------------------------
    print("Running sentiment analysis...")
    sentiments = df_events["analysis_text_en"].apply(extract_sentiment)
    df_events["emotion_label"] = sentiments.apply(lambda x: x[0])
    df_events["sentiment_numeric"] = sentiments.apply(lambda x: x[1])

    # --------------------------------------------------------
    # Step G: ACLED classification
    # --------------------------------------------------------
    print("Classifying ACLED event types...")
    df_events["acled_event_type"] = df_events["analysis_text_en"].apply(
        classify_acled_event
    )

    # --------------------------------------------------------
    # Step H: Narrative topic modeling
    # --------------------------------------------------------
    print("Running BERTopic on events...")
    vectorizer = CountVectorizer(stop_words="english", min_df=3)
    topic_model = BERTopic(vectorizer_model=vectorizer)
    topics, _ = topic_model.fit_transform(
        df_events["analysis_text_en"].tolist()
    )

    df_events["narrative_topic_id"] = topics

    topic_info = topic_model.get_topic_info()
    topic_info.to_csv("conflict_narrative_info.csv", index=False)

    # --------------------------------------------------------
    # Step I: Final export
    # --------------------------------------------------------
    print("Saving event-level dataset...")
    if "publishedAt" in df_events.columns:
        df_events.drop(columns=["publishedAt"], inplace=True)
    df_events.to_csv(OUTPUT_EVENTS, index=False, encoding="utf-8-sig")

    print("✅ Pipeline completed successfully.")

In [9]:
if __name__ == "__main__":
    run_pipeline()

Loading articles...
Detecting geographic scope and extracting locations...
Clustering articles into events...
Reducing to event-level dataset...
Translating event texts...
Running sentiment analysis...
Classifying ACLED event types...
Running BERTopic on events...
Saving event-level dataset...
✅ Pipeline completed successfully.
